# 07 SHAP Analysis

SHAP is used to explain how the final application-time XGBoost model arrives at risk estimates.


In [ ]:
from pathlib import Path
import json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display, Image

ROOT = Path.cwd()
REPORTS = ROOT / 'reports'
MODELS = ROOT / 'models'
DATA_PATH = next((ROOT / 'data' / 'raw').glob('*'))
pd.set_option('display.max_columns', 100)
sns.set_theme(style='whitegrid')
def load_json(path: Path):
    with open(path, 'r', encoding='utf-8') as handle:
        return json.load(handle)

def show_image_if_exists(path: Path, width: int = 900):
    if path.exists():
        display(Image(filename=str(path), width=width))
    else:
        print(f'Missing image: {path}')

from src.utils import load_model

model = load_model(MODELS / 'xgboost_application.pkl')
feature_names = model.named_steps['preprocessor'].get_feature_names_out()
importances = pd.DataFrame({
    'feature': feature_names,
    'xgboost_importance': model.named_steps['classifier'].feature_importances_,
}).sort_values(by='xgboost_importance', ascending=False)
display(importances.head(10))


## Global explanation

The SHAP summary plot shows which features most strongly move default risk predictions across the test sample. For the clean application-time model, bureau score and loan-burden style variables are expected to dominate.


In [ ]:
show_image_if_exists(REPORTS / 'explainability_reports' / 'application_model' / 'xgboost_application_shap_summary.png')
show_image_if_exists(REPORTS / 'explainability_reports' / 'application_model' / 'xgboost_application_shap_local.png')


## Interpretation

Lower bureau score, higher loan burden, and weaker income coverage generally push predictions toward higher risk. Stronger applicant credit quality and more manageable repayment burden pull risk downward.
